# Using an SLM with HuggingFace 

We are going to use the Transformers library to download and interact with a small language model. 


 
### Pre-requisites for this notebook: 

You should already have the necessary libaries installed, but if not, you can install them with (preferably within your virtualenv): 

```bash
$ pip install transformers torch
````


---

## Loading an SLM with the transformers library

First, let's load a model: 

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "microsoft/Phi-3.5-mini-instruct"                 # name of the SLM we wish to use
model = AutoModelForCausalLM.from_pretrained(model_name)       # loads the pretrained model (see below)


Now, we will create a tokenizer and encode some input text to tokens: 

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)          # create the tokenizer (see below)

input_text = "Once upon a time"                                # provide some input text
input_ids = tokenizer.encode(input_text, return_tensors="pt")  # convert the input to tokens as a pytorch tensor (see below)

input_ids

Now feed the tokens into the model to generate an output: 

In [ ]:
output = model.generate(input_ids, max_length=50)              # generate output as tokens

output

You should be looking at the generated token IDs.  

(If you also got a warning about'attention mask not being set, don't worry. We will cover that later). 

Now you have to decode the generated output back to text: 

In [ ]:
decoded_text = tokenizer.decode(output[0], skip_special_tokens=True)
print(decoded_text)

## Using an inference pipeline 

Transformers also comes with pipelines.  From the official docs: 

>
>  Pipelines are a great and easy way to use models for inference. These pipelines are objects that abstract most of the complex code from the library, offering a simple API dedicated to several tasks. 
>
> The pipeline abstraction is a wrapper around all the other available pipelines. It is instantiated as any other pipeline but can provide additional quality of life.
>

Reference: https://huggingface.co/docs/transformers/en/main_classes/pipelines

i.e. the tokenizer makes your life easier by taking care of the tokenization (splitting input text into tokens), encoding, generating an output from the model and decoding the output back to human-readable text. 

Let's see how it works.  First, create the pipeline: 

In [ ]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device='cpu'
)

generator("Once upon a time...", max_length=40, truncation=True)


Notes: 

- It may take a long time to generate an output depending on the model you're running.
- We set the device to "cpu" when creating the pipeline. If you have an Nvidia GPU, try setting it to "cuda" instead.  If you have Apple silicon, try setting to "mps".   
- See the `get_device` function in [model configuration](./07-model-and-tokenizer-config.ipynb) to automatically set the device type. 

🔥 Try customizing the output

In the above, you used a 'max_length' of 40 tokens.  Try customizing this and other parameters: 
- `max_length`: Limit the length of generated text.
- `temperature`: Control randomness, usually ranging from 0 to 1 or 0 to 2, depending  on the model, with higher = more random. You need to set `do_sample` to 'True' if you are using this parameter.
- `top_p`: Nucleus sampling for diverse responses.
- `do_sample`: Enable/disable sampling, set to `True` if you are using `temperature`. This is because without sampling, only the best token is picked, which defeats the point of `temperature` (which makes the generation more random) 

More info on the available parameters can be found in the [documentation](https://huggingface.co/transformers/v3.1.0/main_classes/model.html#transformers.generation_utils.GenerationMixin.generate)

## 🥳 Congratulations 

Well done - you have (hopefully) just interacted an SLM using the world famous Transformers library ! 

You've just loaded a model, used a tokenizer & a pipeline 👏

You can either continue to the next notebook or dig into the details (below) a bit more, to understand what the code is doing... 

-----

## What is a 'causal' language model? 

That "AutoModelForCausalLM" looked pretty scary 🙀 if you haven't seen it before but no need to worry. 

A Causal language model is a model that generates text one token at a time, where each token only depends on the previous tokens. It cannot see future tokens during training or inference.  i.e. it's unidirectional processing. 

💡 Key idea:  “Causal” means the model predicts the next token using only past context (unidirectional processing - where it only sees tokens on the left.)

The alternative to a Causal language model is a "Masked" language model.  
Reference https://huggingface.co/docs/transformers/en/tasks/language_modeling 

### Example:
If you train a causal model on the sentence: 

"The cat sat on the mat."

The model learns to predict each word based only on previous words:

Step	| Input	                | Model predicts
--------|-----------------------|---------------
1	    | "The"	                | "cat"       
2	    | "The cat"	            | "sat"
3	    | "The cat sat"	        | "on"
4   	| "The cat sat on"	    | "the"
5	    | "The cat sat on the"	| "mat"

🚀 Each prediction uses only past information, never future words.

## Loading a pretrained language model 

`AutoModelForCausalLM.from_pretrained(model_name)` loads a model from Hugging Face.   

It downloads it only once and then caches it locally. The next time you load the same model, it will be loaded from the cache instead of downloading again.

The default cache location is:
- Linux/macOS: `~/.cache/huggingface/hub` 
- Windows: `C:\Users\<username>\.cache\huggingface\hub\`



## What is tokenization? 

Tokenization is the process of converting text into a format that the model can understand. It breaks down the text into smaller units, such as words or subwords, and assigns each unit a unique identifier. These unique identifiers are then used as input for the model during training or inference.

![](images/tokenization.png)

<span style="color: green;">**QUESTION**</span>: Can we take the tokenizer that came with model A and use it for model B?

<details>
    <summary>Click to show the answer</summary>

Short answer: You can, but you shouldn't.

Here are the reasons why you shouldn't':

- Different interpretation of words: Recall in the previous section, each word is mapped to a unique identifier. If the tokenizer is different, the model will interpret the input differently. For example, the word "good" might be mapped to token 123 in one tokenizer and token 456 in another tokenizer. This can lead to inconsistencies in the model's predictions.

- Incompatibility: Different models may have different tokenization schemes. If the tokenizer is not compatible with the model's architecture, it can lead to errors or suboptimal performance. For example, some models might expect specific token IDs or special tokens that are not present in the new tokenizer.

- Vocabulary Mismatch: The tokenizer's vocabulary should match the model's expected input. If the tokenizer's vocabulary is different, it can lead to issues in encoding and decoding the input and output. This can result in the model not recognizing certain tokens or producing unexpected outputs.

- Loss of Context: Tokenizers are often designed to capture specific patterns and structures in the training data (defined in the chat templates, which we will cover in `06-chat-templates`). Using a different tokenizer might lead to a loss of context or nuances that the model relies on to generate accurate and coherent outputs.

And here is one reason why you can:

- Some models share the same tokenization scheme, vocabulary, and architecture. In this case, you can use the same tokenizer for different models without any issues. For example, BERT and DistilBERT: Both of these models use the WordPiece tokenizer. DistilBERT is a smaller, faster, and cheaper version of BERT, and it retains the same tokenization strategy.

</details>


## Why do we encode with 'return_tensors' ? 

The `return_tensors="pt"` argument in your `tokenizer.encode()` call ensures that the output is returned as a PyTorch tensor (pt stands for PyTorch).

Normally, tokenizers return a Python list of token IDs, like:

```python 
tokenizer.encode("Once upon a time")
# Output: [7592, 2116, 2203, 50256]    
```

However, when you use return_tensors="pt", the output is a PyTorch tensor:

```python 
tokenizer.encode("Once upon a time", return_tensors="pt")
# Output: tensor([[ 7592, 2116, 2203, 50256]])
```

We do this because language models expect tensors as input, not plain Python lists. 